In [1]:
import pandas as pd
import re
import datetime
import logging
import pathlib
import global3d_track as g3d

import matplotlib.pyplot as plt

# datadir = pathlib.Path('/work/bb1153/b382635/data/track_statistics/dcc_frozen_1e4/tropics')
datadir = pathlib.Path(f'/work/bb1153/b382635/data/track_statistics/grl/resubmission')
outdir = pathlib.Path(f'/work/bb1153/b382635/plots/model_paper/grl_resubmission/data/statistics_ABH')

outdir.mkdir(parents=1, exist_ok=1)

In [8]:
# import os
# os.system("rm /work/bb1153/b382635/plots/model_paper/grl_resubmission/data/statistics_ABH/*")


0

### load and aggregate key results

#### PROCESS

In [3]:
import new_statistics

In [4]:
files = list(datadir.glob('*'))
N = len(files)
print(N, 'DCCs ready for the making')

5369 DCCs ready for the making


In [5]:
# test file names
regex = r'_(\d+)\.nc$'
df = pd.read_csv('/work/bb1153/b382635/data/final_tracks/dcc_frozen_1e4/tropics/track_stats/dcc_validity.csv', index_col=0)
valid_dccs = df.index[df.valid | df.valid_but_complex_convection]
gfiles = []
misc = []
for f in files:
    try:
        dcc = int(re.search(regex, pathlib.Path(f).name).group(1))
        if dcc in valid_dccs:
            gfiles.append(f)
    except:
        misc.append(f)
print(len(gfiles), 'valid DCCs')

5367 valid DCCs


In [9]:
SIZE = 50

for i in range(1,int(N/SIZE)+2):
    # start batch
    logging.info('batch %s' %i)
    t0 = datetime.datetime.now()
    batch = i
    results_path = outdir / f'b{batch}s{SIZE}.nc'
    data_params = dict(files=gfiles, batch=batch, size=SIZE, regex=regex)

    # exists
    if results_path.exists():
        continue

    # calculate
    stats = g3d.analysis.helpers.load_stats(apply=new_statistics.calculate_all, **data_params)
    stats["system"] = stats["system"].assign_attrs(long_name="unique identifier for a tracked deep convective cloud")
    # stats["core"] = stats["core"].assign_attrs(long_name="unique identifier for a convective core associated with a tracked deep convective cloud")

    logging.info('calculated')

    # save
    stats.to_netcdf(results_path)
    tdelta = datetime.datetime.now() - t0
    logging.info('took %s, saved to %s' %(tdelta, results_path))
    stats.close()

INFO:root:batch 1
INFO:root:calculated
INFO:root:took 0:00:04.572549, saved to /work/bb1153/b382635/plots/model_paper/grl_resubmission/data/statistics_ABH/b1s50.nc
INFO:root:batch 2
INFO:root:calculated
INFO:root:took 0:00:04.384886, saved to /work/bb1153/b382635/plots/model_paper/grl_resubmission/data/statistics_ABH/b2s50.nc
INFO:root:batch 3
INFO:root:calculated
INFO:root:took 0:00:04.035419, saved to /work/bb1153/b382635/plots/model_paper/grl_resubmission/data/statistics_ABH/b3s50.nc
INFO:root:batch 4
INFO:root:calculated
INFO:root:took 0:00:04.053233, saved to /work/bb1153/b382635/plots/model_paper/grl_resubmission/data/statistics_ABH/b4s50.nc
INFO:root:batch 5
INFO:root:calculated
INFO:root:took 0:00:03.938538, saved to /work/bb1153/b382635/plots/model_paper/grl_resubmission/data/statistics_ABH/b5s50.nc
INFO:root:batch 6
INFO:root:calculated
INFO:root:took 0:00:04.470357, saved to /work/bb1153/b382635/plots/model_paper/grl_resubmission/data/statistics_ABH/b6s50.nc
INFO:root:batch 

In [6]:
import xarray as xr
import new_statistics
ds = xr.open_dataset(gfiles[5])
r = new_statistics.calculate_all(ds)

In [7]:
r

<xarray.Dataset> Size: 3kB
Dimensions:         (level_full: 51)
Coordinates:
    system          float64 8B 1.52e+04
    lat             float64 8B -12.0
    lon             float64 8B 98.8
  * level_full      (level_full) int32 204B 40 41 42 43 44 45 ... 86 87 88 89 90
Data variables: (12/18)
    ABH             float64 8B ...
    lifetime        float64 8B 4.25
    ncores          int64 8B 2
    ocean_fraction  float32 4B ...
    TOI             datetime64[ns] 8B 2021-07-03T06:20:17
    conv_lifetime   float64 8B 2.25
    ...              ...
    max_CMT_z       (level_full) float64 408B nan nan nan nan ... nan nan nan
    mean_w_z        (level_full) float64 408B nan nan nan nan ... nan nan nan
    cloud_fwc       (level_full) float32 204B nan nan nan nan ... nan nan nan
    cloud_fw        (level_full) float64 408B 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    cloud_area_z    (level_full) float64 408B 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    cloud_cbh_mu    float32 4B 5.463e+03